# ⚖️ Deterministic vs Model-Based Guardrails

Guardrails can be implemented using different approaches depending on how we want to detect unsafe or invalid content.

The two main approaches are:

1. **Deterministic Guardrails**
2. **Model-Based Guardrails**

---

## 🧩 1. Deterministic Guardrails

Deterministic guardrails use **predefined rules** to decide whether content should be allowed or blocked.

Common techniques include:

- Keyword matching
- Regular expressions (Regex)
- Explicit conditions
- Pattern matching
- Rule-based validation

### Example

```text
User Input
    ↓
Rule / Keyword Check
    ↓
Contains banned keyword?
    ├── Yes → 🚫 Block
    └── No  → ✅ Allow

---

## ✅ Advantages of Deterministic Guardrails

- **Fast** — Rule checks are usually very quick.
- **Predictable** — The same input produces the same result.
- **Cost-effective** — No additional LLM call is required.
- **Easy to implement** — Simple rules can be written with Python, Regex, or conditions.
- **Easy to debug** — You can clearly understand why something was blocked.

---

## ❌ Limitations of Deterministic Guardrails

- May miss **nuanced violations**.
- Keyword matching can produce false positives.
- Users can sometimes bypass simple keyword filters.
- Understanding context is difficult.
- Maintaining large rule sets can become complicated.

## 💻 Deterministic Guardrail Example

In [1]:
import re

def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""

    banned_keywords = [
        "hack",
        "exploit",
        "malware",
        "bomb"
    ]

    return any(
        keyword in text.lower()
        for keyword in banned_keywords
    )

In [2]:
test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

for inp in test_inputs:
    blocked = deterministic_guardrail(inp)

    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"

    print(f"{status}: {inp}")

🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


---

## 🤖 2. Model-Based Guardrails

Model-based guardrails use an **LLM or classifier** to evaluate content based on its meaning and context.

Instead of checking only for specific words, the model can reason about what the user is actually asking.

### Example

```text
User Input
    ↓
Safety Model / Classifier
    ↓
Safe or Unsafe?
    ├── SAFE   → ✅ Allow
    └── UNSAFE → 🚫 Block

---

## ✅ Advantages of Model-Based Guardrails

- Can understand **context and meaning**.
- Can detect more **subtle or nuanced issues**.
- More flexible than simple keyword matching.
- Can evaluate complex natural-language requests.
- Useful when simple rules are not sufficient.

---

## ❌ Limitations of Model-Based Guardrails

- **Slower** because an additional model call may be required.
- **More expensive** because model inference consumes tokens/resources.
- Results may not always be perfectly predictable.
- The safety model itself can make mistakes.
- Requires careful prompt/model design.

## 💻 Model-Based Guardrail Example

In [4]:
from langchain_groq import ChatGroq

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety."""

    model = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0
    )

    prompt = f"""
Is the following user input safe to process?

Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}
"""

    result = model.invoke([
        {
            "role": "user",
            "content": prompt
        }
    ])

    return result.content.strip()

In [5]:
for inp in test_inputs:

    verdict = model_based_guardrail(inp)

    status = (
        "🚫 UNSAFE"
        if "UNSAFE" in verdict
        else "✅ SAFE"
    )

    print(f"{status}: {inp}")

🚫 UNSAFE: How do I hack into a database?
✅ SAFE: What is the capital of France?
🚫 UNSAFE: Explain how malware spreads


---

# ⚖️ Deterministic vs Model-Based

| Feature                   | Deterministic      | Model-Based                   |
| ------------------------- | ------------------ | ----------------------------- |
| **Approach**              | Rules and patterns | LLM / Classifier              |
| **Context understanding** | Limited            | Stronger                      |
| **Speed**                 | Fast               | Slower                        |
| **Cost**                  | Low                | Higher                        |
| **Predictability**        | High               | Lower                         |
| **Complexity**             | Simple             | More complex                  |
| **Nuanced detection**     | Limited            | Better                        |
| **Maintenance**           | Rules need updates | Model/prompt needs evaluation |
| **Best for**              | Explicit rules     | Semantic understanding        |

---

## 🧱 When Should You Use Which?

### Use Deterministic Guardrails When:

- You have clear rules.
- You need predictable behavior.
- You need very fast validation.
- You are checking specific patterns.
- You want to avoid additional LLM costs.

### Use Model-Based Guardrails When:

- Context matters.
- Rules are difficult to express with keywords.
- You need semantic understanding.
- You need to detect subtle or nuanced violations.

---

## 💡 Key Idea

> **Deterministic guardrails are rule-based, while model-based guardrails use AI to understand context and meaning.**

Neither approach is universally better.

The right choice depends on the application's requirements for:

- Speed
- Cost
- Predictability
- Context understanding
- Safety
- Complexity

---

## 📝 Key Takeaways

- **Deterministic guardrails** use predefined rules, patterns, keywords, or Regex.
- They are **fast, predictable, and cost-effective**.
- They may miss **contextual or nuanced violations**.
- **Model-based guardrails** use LLMs or classifiers to evaluate content.
- They provide better **semantic and contextual understanding**.
- They are generally **slower and more expensive**.
- Production systems can combine both approaches for **layered protection**.